# ARC3 Meta-Replay Fine-Tuning (Colab)

This notebook mounts Google Drive, prepares the ARC3 + MuZero repos, checks the exported meta-replay corpus, and launches Stage 2 fine-tuning using a precomputed Stage 1 checkpoint.

Current clean replay corpus expected by default:
- `sp80`
- `tr87`
- `vc33`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib, subprocess, sys, shutil, urllib.request, zipfile

WORK = pathlib.Path('/content')
ARC3_DIR = WORK / 'arc3'
MUZERO_DIR = WORK / 'muzero-general'

# Update these if your Drive layout differs.
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/arc3')
ARC3_REC_DIR = DRIVE_ROOT / 'meta_replays'
STAGE1_CKPT = DRIVE_ROOT / 'stage1_pretrain.checkpoint'
OUT_DIR = DRIVE_ROOT / 'colab_meta_replay_results'

def fetch_repo_archive(repo_slug: str, branch: str, target: pathlib.Path) -> None:
    if target.exists():
        print(f'Removing existing directory {target} to ensure a fresh download.')
        shutil.rmtree(target)

    zip_url = f'https://codeload.github.com/{repo_slug}/zip/refs/heads/{branch}'
    zip_path = WORK / f'{target.name}-{branch}.zip'
    extracted_dir = WORK / f'{target.name}-{branch}'

    if zip_path.exists():
        zip_path.unlink()
    if extracted_dir.exists():
        shutil.rmtree(extracted_dir)

    print('Downloading:', zip_url)
    urllib.request.urlretrieve(zip_url, zip_path)
    print('Saved archive:', zip_path, zip_path.stat().st_size, 'bytes')

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(WORK)

    shutil.move(str(extracted_dir), str(target))
    print('Ready:', target)

fetch_repo_archive('humanaiconvention/arc3', 'master', ARC3_DIR)
fetch_repo_archive('werner-duvaud/muzero-general', 'master', MUZERO_DIR)

subprocess.run([
    'pip', 'install', '-q', 'torch', 'torchvision', 'ray[default]', 'tensorboard', 'Pillow', 'numpy'
], check=True)

shutil.copy(ARC3_DIR / 'muzero_patch' / 'models.py', MUZERO_DIR / 'models.py')

for p in [str(MUZERO_DIR), str(ARC3_DIR / 'neurosym')]:
    if p not in sys.path:
        sys.path.insert(0, p)

OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ARC3 repo:', ARC3_DIR)
print('MuZero repo:', MUZERO_DIR)
print('Replay dir:', ARC3_REC_DIR)
print('Stage1 checkpoint:', STAGE1_CKPT)
print('Output dir:', OUT_DIR)

In [ ]:
import torch
print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1e9:.1f} GB)')

In [ ]:
assert ARC3_REC_DIR.exists(), f'Missing replay directory: {ARC3_REC_DIR}'
assert STAGE1_CKPT.exists(), f'Missing Stage 1 checkpoint: {STAGE1_CKPT}'

print('Replay recordings:')
for p in sorted(ARC3_REC_DIR.glob('*.recording.jsonl')):
    print(' ', p.name, p.stat().st_size, 'bytes')

manifest = ARC3_REC_DIR / 'meta_replay_manifest.json'
if manifest.exists():
    print('\nManifest found:', manifest)

In [ ]:
from data_pipeline import ARC3Dataset

arc3_ds = ARC3Dataset(str(ARC3_REC_DIR))
histories = arc3_ds.load_all(verbose=True)
print('\nLoaded histories:', len(histories))
print('Lengths:', [len(h.observation_history) for h in histories])
print('Boundaries:', [h.level_boundaries for h in histories])

## Launch Stage 2 Fine-Tuning

Recommended starting settings for the current small replay corpus:
- `stage2_steps = 15000`
- `batch = 256`
- `use_compile = True`


In [ ]:
stage2_steps = 15000
batch = 256
use_compile = True

In [ ]:
cmd = [
    'python', str(ARC3_DIR / 'scripts' / 'training' / 'colab_stage2_meta_replays.py'),
    '--arc3-dir', str(ARC3_REC_DIR),
    '--stage1-ckpt', str(STAGE1_CKPT),
    '--muzero-dir', str(MUZERO_DIR),
    '--out-dir', str(OUT_DIR),
    '--stage2-steps', str(stage2_steps),
    '--batch', str(batch),
]
if use_compile:
    cmd.append('--compile')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(ARC3_DIR), check=True)

In [ ]:
print('Recent output files:')
for p in sorted(OUT_DIR.rglob('*'), key=lambda x: x.stat().st_mtime, reverse=True)[:20]:
    if p.is_file():
        print(p)